**Task 1**

In [41]:
import json
import math
import os
import statistics
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

FULL = False   # True yaparsan ek olarak n_head = 1, 2, 4, 8, 16 x 3 seed taraması yapar (~15 dk CPU)

In [42]:
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3
n_embd = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [43]:
HAFTA7_BIGRAM_VAL = 2.4907      # hafta 7 bigram_baseline.json
HAFTA7_TEK_HEAD_VAL = 2.4047    # hafta 7 tek_head_sonuc.json

In [44]:
# --- veri, tokenizer, split (hafta 7 ile aynı) ---
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

In [45]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [46]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [47]:

@torch.no_grad()
def pozisyon_bazinda_loss(model, d):
    # Hafta 7'deki gibi: split'in tamamı, örtüşmeyen block_size'lık chunk'lar.
    # t'inci eleman: t+1 karakterlik bağlamla yapılan tahminlerin ortalama loss'u.
    # Ortalaması = val setinin tamamı üzerinden kesin val loss.
    model.eval()
    n_chunks = (len(d) - 1) // block_size
    x = d[:n_chunks * block_size].view(n_chunks, block_size)
    y = d[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    toplam = torch.zeros(block_size)
    for i in range(0, n_chunks, 1024):
        xb, yb = x[i:i + 1024].to(device), y[i:i + 1024].to(device)
        logits, _ = model(xb)
        l = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1), reduction='none')
        toplam += l.view(-1, block_size).sum(0).cpu()
    model.train()
    return toplam / n_chunks

In [49]:
class Head(nn.Module):
    """Tek self-attention head (hafta 7'deki, değişmedi)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                               # (B, T, hs)
        q = self.query(x)                                             # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5           # (B, T, T), head_size ile ölçekle
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)                                    # (B, T, hs)


In [50]:

class MultiHeadAttention(nn.Module):
    """YENİ: num_heads tane head paralel çalışır, çıktıları kanal ekseninde yan yana konur.

    Tek head'de her token geçmişin TEK bir ağırlıklı ortalamasını alabiliyordu;
    şimdi her head kendi query/key'iyle kendi ağırlıklarını seçiyor -> num_heads farklı ortalama.
    """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)          # (B, T, num_heads * hs) = (B, T, n_embd)


In [51]:

class MultiHeadModel(nn.Module):
    """tok + pos -> MultiHeadAttention -> lm_head.   n_head=1 -> hafta 7'deki HeadliBigram'ın aynısı."""

    def __init__(self, n_head):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd, n_head'e tam bölünmeli"
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)   # örn. 4 head x 8 boyut
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                    # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb
        x = self.sa_heads(x)                                                         # (B, T, C)
        logits = self.lm_head(x)                                                     # (B, T, V)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx


def egit(model, ad, log=True):
    """Hafta 7'deki egit() ile aynı döngü."""
    if log:
        print(f'\n--- {ad} ---')
        print(f'{sum(p.numel() for p in model.parameters()):,} parametre')
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            l = estimate_loss(model)
            if log:
                print(f'  step {it:5d}: train {l["train"]:.4f} | val {l["val"]:.4f}')
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model


def kos(n_head, seed, log=True):
    torch.manual_seed(seed)
    model = MultiHeadModel(n_head).to(device)
    egit(model, f'n_head={n_head} (head_size={n_embd // n_head}), seed={seed}', log)
    val = pozisyon_bazinda_loss(model, val_data).mean().item()
    train = pozisyon_bazinda_loss(model, train_data).mean().item()
    sonuc = {'n_head': n_head, 'head_size': n_embd // n_head, 'seed': seed,
             'val_loss': round(val, 4), 'train_loss': round(train, 4),
             'n_params': sum(p.numel() for p in model.parameters())}
    return model, sonuc

In [52]:

# Eğitim: tek head (hafta 7) ve 4 head
# ---------------------------------------------------------------------------
print(f'=== GÖREV 1 === ({device})')
tek, r1 = kos(1, 1337)
multi, r4 = kos(4, 1337)

# ---------------------------------------------------------------------------
# Sonuç tablosu
# ---------------------------------------------------------------------------
print('\n=== SONUÇ: VAL LOSS (tüm val seti, kesin) ===')
print(f'{"model":<36} {"param":>7} {"val loss":>9} {"train":>8} {"bigrama göre":>13} {"perplexity":>11}')
satirlar = [
    ('Bigram (hafta 7)', '', HAFTA7_BIGRAM_VAL, ''),
    ('Tek head 1x32 (hafta 7, kayıtlı)', f'{r1["n_params"]:,}', HAFTA7_TEK_HEAD_VAL, ''),
    ('Tek head 1x32 (bu script)', f'{r1["n_params"]:,}', r1['val_loss'], f'{r1["train_loss"]:.4f}'),
    ('Multi-head 4x8', f'{r4["n_params"]:,}', r4['val_loss'], f'{r4["train_loss"]:.4f}'),
]
for ad, param, val, train in satirlar:
    print(f'{ad:<36} {param:>7} {val:>9.4f} {train:>8} {val - HAFTA7_BIGRAM_VAL:>+13.4f} {math.exp(val):>11.2f}')

print(f'\nYeniden üretim: bu scriptteki tek head {r1["val_loss"]:.4f}, hafta 7 {HAFTA7_TEK_HEAD_VAL:.4f} '
      f'(fark {r1["val_loss"] - HAFTA7_TEK_HEAD_VAL:+.4f})')
print(f'Multi-head düşüşü: {r1["val_loss"]:.4f} -> {r4["val_loss"]:.4f} = {r1["val_loss"] - r4["val_loss"]:.4f}')

# ---------------------------------------------------------------------------
# Kazanç nereden geliyor: pozisyon bazında val loss
# ---------------------------------------------------------------------------
poz1 = pozisyon_bazinda_loss(tek, val_data)
poz4 = pozisyon_bazinda_loss(multi, val_data)
print('\n=== POZİSYON BAZINDA VAL LOSS ===')
print(f'{"t":>3} {"bağlam":>7} {"tek head":>9} {"4 head":>8} {"fark":>8}')
for t in range(block_size):
    print(f'{t:>3} {t + 1:>5} kr {poz1[t].item():>9.4f} {poz4[t].item():>8.4f} {(poz4[t] - poz1[t]).item():>+8.4f}')

# ---------------------------------------------------------------------------
# Örnek metin
# ---------------------------------------------------------------------------
torch.manual_seed(1337)
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # 0 = '\n'
ornek = decode(multi.generate(context, max_new_tokens=400)[0].tolist())
print("\n4 head'li modelden örnek metin:\n" + ornek)

sonuc = {
    'tek_head': r1, 'multi_head_4': r4,
    'hafta7_tek_head_val': HAFTA7_TEK_HEAD_VAL, 'bigram_val': HAFTA7_BIGRAM_VAL,
    'pozisyon_bazinda': {'tek_head': [round(v, 4) for v in poz1.tolist()],
                         'multi_head_4': [round(v, 4) for v in poz4.tolist()]},
    'ornek_metin': ornek, 'device': device,
}

# ---------------------------------------------------------------------------
# İsteğe bağlı: n_head taraması (FULL = True)
# ---------------------------------------------------------------------------
if FULL:
    print('\n=== n_head TARAMASI (3 seed) ===')
    tum = []
    for nh in [1, 2, 4, 8, 16]:
        for seed in [1337, 1338, 1339]:
            _, r = kos(nh, seed, log=False)
            tum.append(r)
            print(f'  n_head={nh:>2} head_size={n_embd // nh:>2} seed={seed}: val {r["val_loss"]:.4f}')
    ort1 = statistics.mean(r['val_loss'] for r in tum if r['n_head'] == 1)
    print(f'\n{"n_head":>6} {"head_size":>9} | {"ortalama":>8} {"std":>6} | {"tek heade göre":>14}')
    for nh in [1, 2, 4, 8, 16]:
        v = [r['val_loss'] for r in tum if r['n_head'] == nh]
        print(f'{nh:>6} {n_embd // nh:>9} | {statistics.mean(v):>8.4f} {statistics.stdev(v):>6.4f} | '
              f'{statistics.mean(v) - ort1:>+14.4f}')
    sonuc['tarama'] = tum

with open('task1_multihead_sonuc.json', 'w', encoding='utf-8') as f:
    json.dump(sonuc, f, ensure_ascii=False, indent=2)
print('\nkaydedildi: task1_multihead_sonuc.json')


=== GÖREV 1 === (cpu)

--- n_head=1 (head_size=32), seed=1337 ---
7,553 parametre
  step     0: train 4.2000 | val 4.2047
  step   500: train 2.6911 | val 2.7087
  step  1000: train 2.5196 | val 2.5303
  step  1500: train 2.4775 | val 2.4829
  step  2000: train 2.4408 | val 2.4523
  step  2500: train 2.4272 | val 2.4435
  step  3000: train 2.4130 | val 2.4327
  step  3500: train 2.3956 | val 2.4212
  step  4000: train 2.4041 | val 2.3992
  step  4500: train 2.3980 | val 2.4084
  step  4999: train 2.3951 | val 2.4126

--- n_head=4 (head_size=8), seed=1337 ---
7,553 parametre
  step     0: train 4.2248 | val 4.2250
  step   500: train 2.6663 | val 2.6809
  step  1000: train 2.5107 | val 2.5189
  step  1500: train 2.4394 | val 2.4447
  step  2000: train 2.3769 | val 2.3890
  step  2500: train 2.3459 | val 2.3606
  step  3000: train 2.3163 | val 2.3361
  step  3500: train 2.2867 | val 2.3138
  step  4000: train 2.2861 | val 2.2796
  step  4500: train 2.2692 | val 2.2816
  step  4999: train

**Task 2**

In [86]:

import json
import math
import os
import statistics
import urllib.request
import warnings

import torch
import torch.nn as nn
from torch.nn import functional as F

ARA_ADIMLAR = True     # False: sadece görevin istediği 6 satır
SEEDS = [1337]         # gürültüyü görmek için [1337, 1338, 1339] (süre 3 katı)

batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3
n_embd = 32
n_head = 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


# %% Batch ve ölçüm fonksiyonları (hafta 7 ile aynı)
def get_batch(split, bs=None):
    bs = bs or batch_size
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (bs,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss_gorev2(model, n_iters):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(n_iters)
        for k in range(n_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def tam_loss(model, d):
    # Split'in tamamı, örtüşmeyen block_size'lık chunk'lar: rastgelelik yok, her seferinde aynı sonuç.
    model.eval()
    n_chunks = (len(d) - 1) // block_size
    x = d[:n_chunks * block_size].view(n_chunks, block_size)
    y = d[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    toplam, sayi = 0.0, 0
    for i in range(0, n_chunks, 1024):
        xb, yb = x[i:i + 1024].to(device), y[i:i + 1024].to(device)
        logits, _ = model(xb)
        toplam += F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1), reduction='sum').item()
        sayi += yb.numel()
    model.train()
    return toplam / sayi


# %% Model parçaları
class BigramLanguageModel(nn.Module):
    """Hafta 7 Görev 1: tablo -> doğrudan logit."""

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss


class Head(nn.Module):
    """Tek self-attention head (hafta 7, değişmedi)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size, proj=False):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd) if proj else None

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        if self.proj is not None:
            out = self.proj(out)
        return out


class FeedForward(nn.Module):
    """YENİ: attention'dan sonra her token kendi başına hesap yapar (token'lar arası etkileşim yok).

    proj=False: Linear(C, C) + ReLU                         (video 1:24, ilk hali)
    proj=True : Linear(C, m*C) + ReLU + Linear(m*C, C)      (residual adımı; m = mult, videoda 4)
    """

    def __init__(self, mult=1, proj=False):
        super().__init__()
        if proj:
            self.net = nn.Sequential(nn.Linear(n_embd, mult * n_embd), nn.ReLU(), nn.Linear(mult * n_embd, n_embd))
        else:
            self.net = nn.Sequential(nn.Linear(n_embd, n_embd), nn.ReLU())

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """YENİ: Transformer bloğu. Önce iletişim (attention), sonra hesaplama (feedforward)."""

    def __init__(self, n_head, ffwd, residual, proj, mult, layernorm):
        super().__init__()
        self.residual = residual
        self.sa = MultiHeadAttention(n_head, n_embd // n_head, proj)
        self.ffwd = FeedForward(mult, proj) if ffwd else None
        # pre-norm: LayerNorm attention'a ve feedforward'a GİRMEDEN önce (videodaki gibi)
        self.ln1 = nn.LayerNorm(n_embd) if layernorm else nn.Identity()
        self.ln2 = nn.LayerNorm(n_embd) if layernorm else nn.Identity()

    def forward(self, x):
        if self.residual:
            x = x + self.sa(self.ln1(x))          # x = x + f(x): gradient toplama üzerinden kesintisiz akar
            if self.ffwd is not None:
                x = x + self.ffwd(self.ln2(x))
        else:
            x = self.sa(self.ln1(x))
            if self.ffwd is not None:
                x = self.ffwd(self.ln2(x))
        return x


class KucukGPT(nn.Module):
    """tok + pos -> n_layer x Block -> (ln_f) -> lm_head.

    n_layer=1, residual=False iken Görev 1'deki model (+ istenirse feedforward). Katmanlar Görev 1 ile aynı
    sırayla kuruluyor: aynı seed -> tek head 2.4047, 4 head 2.2693 aynen çıkar.
    """

    def __init__(self, n_head=4, n_layer=1, ffwd=False, residual=False, proj=False, mult=1, layernorm=False):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_head, ffwd, residual, proj, mult, layernorm) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) if layernorm else nn.Identity()
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(tok_emb + pos_emb)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx



# %% Eğitim döngüsü
def egit_gorev2(model, ad, lr, iters, ev_interval):
    print(f'\n--- {ad} ---  {sum(p.numel() for p in model.parameters()):,} parametre')
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for it in range(iters):
        if it % ev_interval == 0 or it == iters - 1:
            l = estimate_loss_gorev2(model, eval_iters)
            print(f'  step {it:5d}: train {l["train"]:.4f} | val {l["val"]:.4f}')
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model


def kos_gorev2(ad, ayar, seed):
    torch.manual_seed(seed)
    if ayar is None:                                   # bigram, hafta 7 ayarlarıyla
        get_batch('train', bs=4)                       # hafta 7 Görev 1'deki örnek batch + sanity check batch'i
        model = BigramLanguageModel().to(device)       # RNG'den aynı sayıları tüketsin diye -> 2.4907 aynen
        get_batch('train')
        egit_gorev2(model, ad, lr=1e-2, iters=3000, ev_interval=300)
    else:
        model = KucukGPT(**ayar).to(device)
        egit_gorev2(model, ad, lr=learning_rate, iters=max_iters, ev_interval=eval_interval)
    return model, {'val': tam_loss(model, val_data), 'train': tam_loss(model, train_data),
                   'n_params': sum(p.numel() for p in model.parameters())}


# %% Satırlar ve eğitim
# (ad, ★ görevde istenen mi, ayar)   ayar=None -> bigram
SATIRLAR = [
    ('bigram', True, None),
    ('tek head (1x32)', True, dict(n_head=1)),
    ('multi-head (4x8)', True, dict()),
    ('+ feedforward', True, dict(ffwd=True)),
]
if ARA_ADIMLAR:
    SATIRLAR += [
        ('  3 blok, residual yok', False, dict(n_layer=3, ffwd=True)),
        ('  3 blok + proj, residual yok', False, dict(n_layer=3, ffwd=True, proj=True)),
        ('  + residual (ffwd 1x)', False, dict(n_layer=3, ffwd=True, proj=True, residual=True)),
    ]
SATIRLAR += [
    ('+ residual (3 blok, proj, ffwd 4x)', True, dict(n_layer=3, ffwd=True, proj=True, residual=True, mult=4)),
    ('+ LayerNorm', True, dict(n_layer=3, ffwd=True, proj=True, residual=True, mult=4, layernorm=True)),
]

print(f'=== GÖREV 2 === ({device}), seeds={SEEDS}')
sonuclar = []
son_model = None
for ad, istenen, ayar in SATIRLAR:
    rs = []
    for seed in SEEDS:
        model, r = kos_gorev2(f'{ad.strip()}, seed={seed}', ayar, seed)
        rs.append(r)
        print(f'  => val (tüm set) {r["val"]:.4f} | train (tüm set) {r["train"]:.4f}')
    sonuclar.append({'model': ad.strip(), 'istenen': istenen, 'n_params': rs[0]['n_params'],
                     'val_seedler': [round(r['val'], 4) for r in rs],
                     'val': statistics.mean(r['val'] for r in rs),
                     'train': statistics.mean(r['train'] for r in rs),
                     'std': statistics.stdev(r['val'] for r in rs) if len(rs) > 1 else None})
    son_model = model


# %% Tablo
print('\n=== GÖREV 2: VAL LOSS (val setinin tamamı, kesin) ===   ★ = görevde istenen satır')
print(f'{"":2}{"model":<36} {"param":>7} {"val":>8} {"train":>8} {"val-train":>10} {"önceki satıra":>14} {"bigrama":>9}')
bigram_val = sonuclar[0]['val']
onceki = None
for s in sonuclar:
    d = '' if onceki is None else f'{s["val"] - onceki:+.4f}'
    std = '' if s['std'] is None else f' ±{s["std"]:.3f}'
    ad = next(a for a, _, _ in SATIRLAR if a.strip() == s['model'])
    print(f'{"★ " if s["istenen"] else "  "}{ad:<36} {s["n_params"]:>7,} {s["val"]:>8.4f}{std} {s["train"]:>8.4f} '
          f'{s["val"] - s["train"]:>+10.4f} {d:>14} {s["val"] - bigram_val:>+9.4f}')
    onceki = s['val']

torch.manual_seed(1337)
ornek = decode(son_model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 400)[0].tolist())
print("\nSon modelden (LayerNorm'lu küçük GPT) örnek metin:\n" + ornek)


# %% LayerNorm ve hafta 6'daki BatchNorm1d
class BatchNorm1d:
    # Hafta 6'daki sınıf, aynen (YZ50_Week6 notebook'u)

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # gradient descent ile öğrenilen parametreler
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # gradient descent ile değil, momentum ile güncellenen istatistikler
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):

        if self.training:

            # 2D input: (B, C)
            if x.ndim == 2:
                dim = 0

            # 3D input: (B, T, C)
            elif x.ndim == 3:
                dim = (0, 1)

            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True)

        else:
            xmean = self.running_mean
            xvar = self.running_var

        # normalize
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)

        # öğrenilebilir scale + shift
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (
                    (1 - self.momentum) * self.running_mean
                    + self.momentum * xmean.squeeze()
                )

                self.running_var = (
                    (1 - self.momentum) * self.running_var
                    + self.momentum * xvar.squeeze()
                )

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class LayerNorm1d:
    # Hafta 6'daki BatchNorm1d'den türetildi. Değişen 3 şey:
    #   1. eksen: batch boyunca (dim 0 ya da (0, 1)) yerine her örneğin kendi feature'ları boyunca (dim=-1)
    #   2. running_mean / running_var silindi
    #   3. training flag'i silindi: eğitimde ve üretimde aynı hesap
    # unbiased=False: nn.LayerNorm ile birebir aynı olsun diye (N'e böler, N-1'e değil)

    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(-1, keepdim=True)
        xvar = x.var(-1, keepdim=True, unbiased=False)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


lnbn = {}   # ölçümleri JSON'a da yazalım
torch.manual_seed(1337)
B, C = 32, 100
x = torch.randn(B, C) * torch.linspace(0.5, 5, C) + torch.linspace(-3, 3, C)   # sütunlar farklı ölçekte

print('\n=== LayerNorm vs BatchNorm1d ===')
print('\n1. HANGİ EKSEN?  x:', tuple(x.shape), '(satır = örnek, sütun = feature)')
ybn, yln = BatchNorm1d(C)(x), LayerNorm1d(C)(x)
print(f'{"":<13} {"sütun ort.":>11} {"sütun std":>10} | {"satır ort.":>11} {"satır std":>10}')
for ad, y in (('girdi', x), ('BatchNorm1d', ybn), ('LayerNorm1d', yln)):
    print(f'{ad:<13} {y.mean(0).abs().mean():>11.4f} {y.std(0).mean():>10.4f} | '
          f'{y.mean(1).abs().mean():>11.4f} {y.std(1).mean():>10.4f}')
h = torch.randn(4, 8, 32) * 3 + 1                                              # GPT girdisi gibi (B, T, C)
hbn, hln = BatchNorm1d(32)(h), LayerNorm1d(32)(h)
print(f'(B, T, C)=(4, 8, 32): BatchNorm her kanalı B*T=32 token boyunca normalize ediyor -> '
      f'kanal ort. {hbn.mean((0, 1)).abs().mean():.4f}, token ort. {hbn.mean(-1).abs().mean():.4f}')
print(f'                      LayerNorm her token\'ı kendi 32 kanalı boyunca normalize ediyor -> '
      f'kanal ort. {hln.mean((0, 1)).abs().mean():.4f}, token ort. {hln.mean(-1).abs().mean():.4f}')

print('\n2. BİR ÖRNEĞİN ÇIKTISI BATCH\'TEKİ DİĞERLERİNE BAĞLI MI?')
batch_a = torch.cat([x[:1], x[1:16]])                       # aynı ilk örnek, farklı arkadaşlar
batch_b = torch.cat([x[:1], torch.randn(15, C) * 10])
lnbn['batch_bagimliligi'] = {'bn': (BatchNorm1d(C)(batch_a)[0] - BatchNorm1d(C)(batch_b)[0]).abs().max().item(),
                              'ln': (LayerNorm1d(C)(batch_a)[0] - LayerNorm1d(C)(batch_b)[0]).abs().max().item()}
print(f'aynı örnek, iki farklı batch -> BatchNorm çıktı farkı: {lnbn["batch_bagimliligi"]["bn"]:.4f}')
print(f'                               LayerNorm çıktı farkı: {lnbn["batch_bagimliligi"]["ln"]:.4f}')

print('\n3. TRAIN / EVAL FARKI')
bn = BatchNorm1d(C)
for _ in range(20):                                         # birkaç eğitim adımı: running stats dolsun
    bn(x[torch.randint(0, B, (16,))])
tr_out = bn(x)[0]
bn.training = False
ev_out = bn(x)[0]
lnbn['train_eval_farki_bn'] = (tr_out - ev_out).abs().max().item()
print(f'BatchNorm, aynı girdi: train modu vs eval modu çıktı farkı: {lnbn["train_eval_farki_bn"]:.4f}')
print('LayerNorm: modu yok, saklanan istatistik yok -> fark tanım gereği 0')

print('\n4. TEK ÖRNEK (batch = 1, örn. üretim anı)')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')                         # "degrees of freedom <= 0" uyarısı beklenen şey
    bn_tek = BatchNorm1d(C)(x[:1])
print(f'BatchNorm (train modu): NaN var mı: {torch.isnan(bn_tek).any().item()}  (tek değerin varyansı tanımsız)')
print(f'LayerNorm             : NaN var mı: {torch.isnan(LayerNorm1d(C)(x[:1])).any().item()}')

print('\n5. GPT GİRDİSİNDE (B, T, C): GELECEKTEN BİLGİ SIZIYOR MU?')
torch.manual_seed(0)
h = torch.randn(4, 8, 32)
h2 = h.clone()
h2[:, -1] += 5.0                                            # sadece SON pozisyonu (geleceği) değiştir
bn_fark = (BatchNorm1d(32)(h)[:, 0] - BatchNorm1d(32)(h2)[:, 0]).abs().max()
ln_fark = (LayerNorm1d(32)(h)[:, 0] - LayerNorm1d(32)(h2)[:, 0]).abs().max()
lnbn['gelecek_sizintisi'] = {'bn': bn_fark.item(), 'ln': ln_fark.item()}
print(f"pozisyon 7 değişince pozisyon 0'ın çıktısı -> BatchNorm farkı: {bn_fark:.4f} | LayerNorm farkı: {ln_fark:.4f}")

print('\n6. LayerNorm1d == nn.LayerNorm ?')
with torch.no_grad():
    ref = nn.LayerNorm(32)(h)
print('allclose:', torch.allclose(ref, LayerNorm1d(32)(h), atol=1e-6))
unb = (h - h.mean(-1, keepdim=True)) / torch.sqrt(h.var(-1, keepdim=True) + 1e-5)
print(f'BatchNorm1d gibi unbiased var (N-1) kullanılsaydı en büyük fark: {(unb - ref).abs().max():.4f}')

with open('task2_sonuc.json', 'w', encoding='utf-8') as f:
    json.dump({'seeds': SEEDS, 'tablo': [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in s.items()}
                                         for s in sonuclar],
               'ln_bn': lnbn, 'ornek_metin': ornek, 'device': device},
              f, ensure_ascii=False, indent=2)
print('\nkaydedildi: task2_sonuc.json')

=== GÖREV 2 === (cpu), seeds=[1337]

--- bigram, seed=1337 ---  4,225 parametre
  step     0: train 4.6352 | val 4.6489
  step   300: train 2.8162 | val 2.8343
  step   600: train 2.5439 | val 2.5779
  step   900: train 2.4925 | val 2.5144
  step  1200: train 2.4853 | val 2.5072
  step  1500: train 2.4664 | val 2.4957
  step  1800: train 2.4682 | val 2.4961
  step  2100: train 2.4701 | val 2.4875
  step  2400: train 2.4639 | val 2.4897
  step  2700: train 2.4729 | val 2.4924
  step  2999: train 2.4611 | val 2.4912
  => val (tüm set) 2.4907 | train (tüm set) 2.4612

--- tek head (1x32), seed=1337 ---  7,553 parametre
  step     0: train 4.2000 | val 4.2047
  step   500: train 2.6911 | val 2.7087
  step  1000: train 2.5196 | val 2.5303
  step  1500: train 2.4775 | val 2.4829
  step  2000: train 2.4408 | val 2.4523
  step  2500: train 2.4272 | val 2.4435
  step  3000: train 2.4130 | val 2.4327
  step  3500: train 2.3956 | val 2.4212
  step  4000: train 2.4041 | val 2.3992
  step  4500: tr

**Task 3**

In [87]:

import os
import time
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- donanıma göre konfigürasyon ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':   # GPU (örn. Colab T4): videodaki konfig
    n_embd, n_head, n_layer, block_size, batch_size = 384, 6, 6, 256, 64
    dropout, learning_rate, max_iters, eval_iters = 0.2, 3e-4, 5000, 200
else:                  # CPU (Colab CPU, 2 çekirdek)
    n_embd, n_head, n_layer, block_size, batch_size = 128, 4, 4, 64, 32
    dropout, learning_rate, max_iters, eval_iters = 0.1, 1e-3, 4000, 50
eval_interval = 500
torch.manual_seed(1337)

# --- veri (önceki görevlerle aynı) ---
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt', 'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch_gorev3(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss_gorev3(model):
    out = {}
    model.eval()                                   # eval(): dropout kapalı
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_gorev3(split)
            losses[k] = model(X, Y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def tam_val_loss_gorev3(model):
    """Val setinin tamamı, örtüşmeyen block_size'lık chunk'lar (önceki görevlerdeki kesin ölçü)."""
    model.eval()
    n_chunks = (len(val_data) - 1) // block_size
    x = val_data[:n_chunks * block_size].view(n_chunks, block_size)
    y = val_data[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    toplam = 0.0
    for i in range(0, n_chunks, 256):
        logits, _ = model(x[i:i + 256].to(device))
        toplam += F.cross_entropy(logits.reshape(-1, vocab_size), y[i:i + 256].reshape(-1).to(device),
                                  reduction='sum').item()
    model.train()
    return toplam / y.numel()


# --- model: Görev 2'nin son hali + dropout ---
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)          # YENİ

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))  # YENİ: bazı token'ların "konuşmasını" rastgele kes
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)          # YENİ

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd),
                                 nn.Dropout(dropout))   # YENİ

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        return x + self.ffwd(self.ln2(x))


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])   # Görev 2'de sabit 3'tü
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        logits = self.lm_head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx


# --- eğitim ---
model = GPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'cihaz: {device} | n_embd {n_embd}, n_head {n_head} (head_size {n_embd // n_head}), n_layer {n_layer}, '
      f'block_size {block_size}, dropout {dropout}, batch {batch_size}, lr {learning_rate}, {max_iters} adım')
print(f'parametre: {n_params:,} ({n_params / 1e6:.2f} M)\n')out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
t0 = time.time()
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        l = estimate_loss_gorev3(model)
        print(f'step {it:5d}: train {l["train"]:.4f} | val {l["val"]:.4f}  ({(time.time() - t0) / 60:.1f} dk)')
    xb, yb = get_batch_gorev3('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
egitim_suresi = time.time() - t0

# --- sonuç ---
val = tam_val_loss_gorev3(model)
print('\n=== SONUÇ ===')
print(f'parametre    : {n_params:,}')
print(f'val loss     : {val:.4f}  (val setinin tamamı)   son tahmini train: {l["train"]:.4f}')
print(f'eğitim süresi: {egitim_suresi / 60:.1f} dk ({1000 * egitim_suresi / max_iters:.0f} ms/adım, {device})')

model.eval()                                       # üretirken dropout kapalı
torch.manual_seed(1337)
ornek = decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist())
print('\nörnek metin:\n' + ornek)


cihaz: cpu | n_embd 128, n_head 4 (head_size 32), n_layer 4, block_size 64, dropout 0.1, batch 32, lr 0.001, 4000 adım
parametre: 816,705 (0.82 M)

step     0: train 4.3339 | val 4.3330  (0.2 dk)
step   500: train 1.9674 | val 2.0300  (3.3 dk)
step  1000: train 1.7209 | val 1.8774  (6.4 dk)
step  1500: train 1.6120 | val 1.7859  (9.5 dk)
step  2000: train 1.5496 | val 1.7463  (12.5 dk)
step  2500: train 1.5054 | val 1.6947  (15.7 dk)
step  3000: train 1.4781 | val 1.6675  (18.8 dk)
step  3500: train 1.4533 | val 1.6491  (21.9 dk)
step  3999: train 1.4326 | val 1.6417  (25.0 dk)

=== SONUÇ ===
parametre    : 816,705
val loss     : 1.6366  (val setinin tamamı)   son tahmini train: 1.4326
eğitim süresi: 25.0 dk (374 ms/adım, cpu)

örnek metin:

Citizen the enemEe of rolius?

NORTHUMBERLAND:
Sir.

First, this termain:
O, see, so take you presain'd more shall come to
But to gone. God, I am poece yong in this father, trutchmen
Marcius pleasure of heaven shielded this, and husbind,
As gone no

**Task 4**

In [89]:


import math
import os
import time
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- donanıma göre konfigürasyon (Görev 3 ile aynı) ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    n_embd, n_head, n_layer, block_size, batch_size = 384, 6, 6, 256, 64
    dropout, learning_rate, max_iters, eval_iters = 0.2, 3e-4, 5000, 200
else:
    n_embd, n_head, n_layer, block_size, batch_size = 128, 4, 4, 64, 32
    dropout, learning_rate, max_iters, eval_iters = 0.1, 1e-3, 4000, 50
eval_interval = 500
torch.manual_seed(1337)

TURKCE_PATH = 'turkish.txt'        # GÖREV 4
SHAKESPEARE_VAL_LOSS = None        # GÖREV 4: Görev 3'te bulduğun val loss'u buraya yaz (örn. 1.75)

# --- GÖREV 4: veri = Türkçe metin; Shakespeare sadece karakter karşılaştırması için ---
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt', 'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    shakespeare = f.read()
with open(TURKCE_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# --- GÖREV 4: karakter karşılaştırması ---
tr_set, en_set = set(text), set(shakespeare)
print('=== KARAKTERLER ===')
print(f'Türkçe     : {len(text):,} karakter, {len(text.encode("utf-8")):,} byte, {len(tr_set)} farklı karakter')
print(f'Shakespeare: {len(shakespeare):,} karakter, {len(shakespeare.encode("utf-8")):,} byte, {len(en_set)} farklı karakter')
print(f'ortak: {len(tr_set & en_set)}')
print(f'sadece Türkçede     : {"".join(sorted(tr_set - en_set))!r}')
print(f'sadece Shakespeare\'de: {"".join(sorted(en_set - tr_set))!r}')

# --- GÖREV 4: val seti train'de görülmemiş metin mi? (val loss'u yorumlamak için) ---
train_satirlar = set(l for l in text[:n].split('\n') if l.strip())
val_satirlar = [l for l in text[n:].split('\n') if l.strip()]
tekrar = sum(l in train_satirlar for l in val_satirlar) / len(val_satirlar)
print(f'\nval satırlarının train\'de birebir geçme oranı: {tekrar:.1%}\n')


def get_batch_gorev4(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss_gorev4(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_gorev4(split)
            losses[k] = model(X, Y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def tam_val_loss_gorev4(model):
    model.eval()
    n_chunks = (len(val_data) - 1) // block_size
    x = val_data[:n_chunks * block_size].view(n_chunks, block_size)
    y = val_data[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    toplam = 0.0
    for i in range(0, n_chunks, 256):
        logits, _ = model(x[i:i + 256].to(device))
        toplam += F.cross_entropy(logits.reshape(-1, vocab_size), y[i:i + 256].reshape(-1).to(device),
                                  reduction='sum').item()
    model.train()
    return toplam / y.numel()


# --- model (Görev 3 ile birebir aynı) ---
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd),
                                 nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        return x + self.ffwd(self.ln2(x))


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        logits = self.lm_head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx


# --- eğitim (Görev 3 ile aynı) ---
model = GPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'cihaz: {device} | n_embd {n_embd}, n_head {n_head}, n_layer {n_layer}, block_size {block_size}, '
      f'dropout {dropout}, vocab {vocab_size}')
print(f'parametre: {n_params:,} ({n_params / 1e6:.2f} M)\n')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
t0 = time.time()
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        l = estimate_loss_gorev4(model)
        print(f'step {it:5d}: train {l["train"]:.4f} | val {l["val"]:.4f}  ({(time.time() - t0) / 60:.1f} dk)')
    xb, yb = get_batch_gorev4('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
egitim_suresi = time.time() - t0

# --- sonuç ---
val = tam_val_loss_gorev4(model)
print('\n=== SONUÇ ===')
print(f'parametre    : {n_params:,}')
print(f'val loss     : {val:.4f}  (val setinin tamamı)   son tahmini train: {l["train"]:.4f}')
print(f'eğitim süresi: {egitim_suresi / 60:.1f} dk ({device})')

# --- GÖREV 4: val loss karşılaştırması ---
print('\n=== VAL LOSS KARŞILAŞTIRMASI ===')
print(f'{"":<28} {"Shakespeare":>12} {"Türkçe":>8}')
print(f'{"farklı karakter (vocab)":<28} {len(en_set):>12} {vocab_size:>8}')
print(f'{"rastgele tahmin, ln(vocab)":<28} {math.log(len(en_set)):>12.4f} {math.log(vocab_size):>8.4f}')
en_val = f'{SHAKESPEARE_VAL_LOSS:.4f}' if SHAKESPEARE_VAL_LOSS else '(Görev 3)'
print(f'{"GPT val loss":<28} {en_val:>12} {val:>8.4f}')

model.eval()
torch.manual_seed(1337)
ornek = decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist())
print('\nörnek metin:\n' + ornek)

=== KARAKTERLER ===
Türkçe     : 994,750 karakter, 1,115,854 byte, 72 farklı karakter
Shakespeare: 1,115,394 karakter, 1,115,394 byte, 65 farklı karakter
ortak: 50
sadece Türkçede     : '012456789ÇÖÜâçöüğİış—’'
sadece Shakespeare'de: "!$&'-:JLQWXjqwx"

val satırlarının train'de birebir geçme oranı: 71.3%

cihaz: cpu | n_embd 128, n_head 4, n_layer 4, block_size 64, dropout 0.1, vocab 72
parametre: 818,504 (0.82 M)

step     0: train 4.3917 | val 4.3899  (0.2 dk)
step   500: train 0.8574 | val 0.8728  (3.3 dk)
step  1000: train 0.3687 | val 0.4120  (6.4 dk)
step  1500: train 0.2794 | val 0.3013  (9.5 dk)
step  2000: train 0.2376 | val 0.2492  (12.6 dk)
step  2500: train 0.2111 | val 0.2373  (15.6 dk)
step  3000: train 0.2041 | val 0.2241  (18.6 dk)
step  3500: train 0.1987 | val 0.2121  (21.7 dk)
step  3999: train 0.1915 | val 0.2031  (24.8 dk)

=== SONUÇ ===
parametre    : 818,504
val loss     : 0.2081  (val setinin tamamı)   son tahmini train: 0.1915
eğitim süresi: 24.9 dk (cpu)

=== 